# Belajar Analisis Sentimen - Dari Dasar

Notebook ini akan memandu kamu memahami cara kerja analisis sentimen step by step.

**Apa itu Analisis Sentimen?**
Teknik NLP (Natural Language Processing) untuk menentukan apakah suatu teks bernada **positif**, **negatif**, atau **netral**.

---
## Step 1: Persiapan Library

Kita akan pakai beberapa library:
- **re** → untuk preprocessing teks (regex)
- **sastrawi** → stemming bahasa Indonesia
- **scikit-learn** → untuk TF-IDF dan model ML
- **textblob** → rule-based sentiment analyzer (untuk perbandingan)

In [ ]:
# Install library yang dibutuhkan (jalankan sekali)
!pip install scikit-learn sastrawi textblob pandas numpy matplotlib seaborn

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sastrawi.stemmer import Stemmer
from sastrawi.stop_words import StopWords

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 12

print('Semua library berhasil di-import!')

---
## Step 2: Siapkan Data

Kita mulai dengan dataset sederhana dulu supaya gampang dipahami.
Nanti bisa diganti dengan dataset yang lebih besar.

In [ ]:
# Dataset contoh: review produk
# Label: 1 = positif, -1 = negatif, 0 = netral

data = {
    'teks': [
        'Produk ini sangat bagus dan berkualitas',
        'Pelayanan cepat dan ramah sekali',
        'Saya sangat puas dengan hasilnya',
        'Barangnya keren, sesuai deskripsi',
        'Luar biasa, recomended banget',
        'Kualitas oke punya, tidak mengecewakan',
        'Pengiriman super cepat, packaging rapi',
        'Harga murah tapi kualitas mantap',
        'Sangat recommended, akan beli lagi',
        'Top markotop, seller terbaik',
        'Produk jelek, kualitas buruk sekali',
        'Pelayanan lambat dan tidak ramah',
        'Kecewa berat, tidak sesuai foto',
        'Barangnya rusak saat sampai',
        'Sangat tidak puas, mau refund',
        'Kualitas mengecewakan, jangan beli',
        'Pengiriman lama banget, parah',
        'Harga mahal tapi kualitas rendah',
        'Penipuan, barang tidak dikirim',
        'Seller tidak bertanggung jawab',
        'Produk biasa saja, tidak istimewa',
        'Standar lah, seperti biasa',
        'Lumayan oke untuk harganya',
        'Barang sampai dengan selamat',
        'Sesuai dengan yang dipesan',
    ],
    'label': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
              -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
              0, 0, 0, 0, 0]
}

df = pd.DataFrame(data)
print(f'Total data: {len(df)}')
print(f'\nDistribusi label:')
print(df['label'].value_counts().map({1: 'Positif', -1: 'Negatif', 0: 'Netral'}))
df

---
## Step 3: Preprocessing Teks

Sebelum teks bisa diproses model ML, kita harus bersihkan dulu:

1. **Case folding** → ubah ke huruf kecil
2. **Cleaning** → hapus angka, tanda baca, URL
3. **Stopword removal** → hapus kata umum yang tidak bermakna (yang, dan, di, ke, dll)
4. **Stemming** → kembalikan kata ke bentuk dasar (membeli → beli)

In [ ]:
stopwords = StopWords()
stemmer = Stemmer()

def preprocess(teks):
    # 1. Case folding
    teks = teks.lower()
    
    # 2. Cleaning: hapus angka, tanda baca, karakter spesial
    teks = re.sub(r'[^a-z\s]', '', teks)
    
    # 3. Hapus extra whitespace
    teks = re.sub(r'\s+', ' ', teks).strip()
    
    # 4. Stopword removal
    kata_kata = teks.split()
    kata_kata = [k for k in kata_kata if k not in stopwords]
    
    # 5. Stemming
    teks = ' '.join([stemmer.stem(k) for k in kata_kata])
    
    return teks

# Terapkan preprocessing
df['teks_bersih'] = df['teks'].apply(preprocess)

# Lihat perbandingan sebelum dan sesudah
comparison = df[['teks', 'teks_bersih']].copy()
comparison.columns = ['Sebelum', 'Sesudah']
comparison

---
## Step 4: Feature Extraction (Mengubah Teks ke Angka)

Model ML tidak bisa memproses teks langsung. Kita harus ubah jadi angka dulu.

### Metode 1: Bag of Words (BoW)
Hitung frekuensi setiap kata di setiap dokumen. Simpel tapi bisa bekerja.

In [ ]:
# Bag of Words
bow = CountVectorizer()
X_bow = bow.fit_transform(df['teks_bersih'])

print(f'Jumlah kata unik (vocabulary): {len(bow.get_feature_names_out())}')
print(f'Ukuran matrix: {X_bow.shape}')
print(f'\nContoh vocabulary (20 kata pertama):')
print(bow.get_feature_names_out()[:20])

# Tampilkan sebagai DataFrame supaya mudah dibaca
df_bow = pd.DataFrame(X_bow.toarray(), columns=bow.get_feature_names_out())
print(f'\nMatrix Bag of Words:')
df_bow.head(10)

### Metode 2: TF-IDF (Term Frequency - Inverse Document Frequency)

Lebih pintar dari BoW. Kata yang sering muncul di **semua** dokumen dapat bobot rendah.
Kata yang khas di satu dokumen saja dapat bobot tinggi.

Rumus:
- **TF** = seberapa sering kata muncul di dokumen ini
- **IDF** = seberapa jarang kata muncul di semua dokumen
- **TF-IDF** = TF × IDF

In [ ]:
# TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df['teks_bersih'])

print(f'Ukuran matrix TF-IDF: {X_tfidf.shape}')

df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())
print(f'\nMatrix TF-IDF (nilai lebih halus, antara 0-1):')
df_tfidf.head(10)

In [ ]:
# Lihat kata dengan bobot TF-IDF tertinggi di dokumen pertama
doc_idx = 0
feature_names = tfidf.get_feature_names_out()
scores = X_tfidf[doc_idx].toarray().flatten()

top_kata = sorted(zip(feature_names, scores), key=lambda x: x[1], reverse=True)[:10]

print(f'Teks: "{df.iloc[doc_idx]["teks"]}"')
print(f'\nKata dengan bobot TF-IDF tertinggi:')
for kata, skor in top_kata:
    print(f'  {kata:20s} → {skor:.4f}')

---
## Step 5: Latih Model ML (Naive Bayes)

Naive Bayes adalah model klasifikasi berbasis probabilitas.
Simpel, cepat, dan cocok untuk klasifikasi teks.

Konsepnya: berdasarkan kata-kata yang muncul, berapa probabilitas teks ini positif vs negatif?

In [ ]:
# Split data: training (80%) dan testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, df['label'], test_size=0.2, random_state=42
)

print(f'Data training: {X_train.shape[0]}')
print(f'Data testing:  {X_test.shape[0]}')

# Latih model Naive Bayes
model = MultinomialNB()
model.fit(X_train, y_train)

# Prediksi data testing
y_pred = model.predict(X_test)

# Evaluasi
print(f'\nAkurasi: {accuracy_score(y_test, y_pred):.2%}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Negatif', 'Netral', 'Positif']))

In [ ]:
# Confusion Matrix - visualisasi performa model
cm = confusion_matrix(y_test, y_pred, labels=[-1, 0, 1])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negatif', 'Netral', 'Positif'],
            yticklabels=['Negatif', 'Netral', 'Positif'])
plt.title('Confusion Matrix')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.tight_layout()
plt.show()

---
## Step 6: Uji dengan Teks Baru

Sekarang kita bisa pakai model yang sudah dilatih untuk menganalisis sentimen teks baru.

In [ ]:
def prediksi_sentimen(teks, model, tfidf_vectorizer):
    # Preprocessing
    teks_bersih = preprocess(teks)
    
    # Transform ke TF-IDF
    X = tfidf_vectorizer.transform([teks_bersih])
    
    # Prediksi
    label = model.predict(X)[0]
    proba = model.predict_proba(X)[0]
    
    label_map = {-1: 'NEGATIF', 0: 'NETRAL', 1: 'POSITIF'}
    
    print(f'Teks:     "{teks}"')
    print(f'Bersih:   "{teks_bersih}"')
    print(f'Sentimen: {label_map[label]}')
    print(f'\nProbabilitas:')
    for i, (lbl, prob) in enumerate(zip([-1, 0, 1], proba)):
        bar = '█' * int(prob * 30)
        print(f'  {label_map[lbl]:10s}: {prob:.2%} {bar}')
    print()

# Coba dengan kalimat baru
kalimat_baru = [
    'Barangnya bagus banget, saya suka!',
    'Pelayanan jelek, lama banget nunggunya',
    'Barang sudah sampai, sesuai pesanan',
    'Kecewa, kualitasnya tidak sesuai harapan',
    'Lumayan lah, tidak terlalu bagus tidak terlalu jelek',
]

for kalimat in kalimat_baru:
    prediksi_sentimen(kalimat, model, tfidf)

---
## Step 7: Analisis Kata Paling Berpengaruh

Kata mana yang paling berkontribusi ke sentimen positif atau negatif?

In [ ]:
# Ekstrak bobot kata dari model Naive Bayes
feature_names = tfidf.get_feature_names_out()

# log_probabilities dari Naive Bayes
log_probs = model.feature_log_prob_

# Untuk setiap kelas, cari kata dengan bobot tertinggi
label_map = {-1: 'Negatif', 0: 'Netral', 1: 'Positif'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (label, name) in enumerate(zip([-1, 0, 1], ['Negatif', 'Netral', 'Positif'])):
    class_idx = list(model.classes_).index(label)
    top_indices = log_probs[class_idx].argsort()[-15:][::-1]
    top_words = feature_names[top_indices]
    top_scores = log_probs[class_idx][top_indices]
    
    colors = ['#e74c3c' if label == -1 else '#95a5a6' if label == 0 else '#2ecc71'] * 15
    axes[idx].barh(range(15), top_scores, color=colors)
    axes[idx].set_yticks(range(15))
    axes[idx].set_yticklabels(top_words)
    axes[idx].set_title(f'Kata Paling Berpengaruh\n({name})')
    axes[idx].set_xlabel('Log Probability')
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

---
## Step 8: Perbandingan dengan Model Lain

Coba bandingkan Naive Bayes dengan model lain.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM (Linear)': LinearSVC(max_iter=10000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

results = []

for nama, mdl in models.items():
    mdl.fit(X_train, y_train)
    y_pred = mdl.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append({'Model': nama, 'Akurasi': acc})
    print(f'{nama:25s} → Akurasi: {acc:.2%}')

# Visualisasi perbandingan
df_results = pd.DataFrame(results)
plt.figure(figsize=(10, 5))
sns.barplot(data=df_results, x='Akurasi', y='Model', palette='viridis')
plt.title('Perbandingan Akurasi Model')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

---
## Ringkasan: Alur Analisis Sentimen

```
Teks Mentah
    ↓
Preprocessing (case fold, cleaning, stopword, stemming)
    ↓
Feature Extraction (BoW / TF-IDF / Word Embedding)
    ↓
Model ML (Naive Bayes / SVM / LSTM / BERT)
    ↓
Prediksi Sentimen (Positif / Negatif / Netral)
```

## Langkah Selanjutnya

- Coba dengan dataset yang lebih besar (misal: review dari Tokopedia/Shopee)
- Eksperimen dengan Word Embedding (Word2Vec, GloVe, FastText)
- Coba Deep Learning (LSTM, BERT) untuk akurasi lebih tinggi
- Deploy jadi web app (Flask/FastAPI + React)